In [ ]:
# Copy the prepared atlas from Google Drive to local Colab disk.
from google.colab import drive
from pathlib import Path
import shutil
import h5py
from tqdm.auto import tqdm

drive.mount('/content/drive')

src = Path('/content/drive/MyDrive/senescence_atlas_processed.h5')
dst = Path('/content/senescence_atlas_processed.h5')
partial = dst.with_suffix('.h5.partial')
# HDF5 byte size can differ across library versions after preprocessing.
# Require a completed prepared atlas and use its actual size for the copy.
with h5py.File(src, 'r') as prepared:
    assert bool(prepared.attrs.get('complete', False)), 'Preprocessing is incomplete.'
expected = src.stat().st_size


with src.open('rb') as reader, partial.open('wb') as writer:
    with tqdm(total=expected, unit='B', unit_scale=True, desc='Copying atlas') as bar:
        while chunk := reader.read(32 * 1024**2):
            writer.write(chunk)
            bar.update(len(chunk))

assert partial.stat().st_size == expected
partial.replace(dst)

print(f"Free disk: {shutil.disk_usage('/content').free / 1e9:.1f} GB")
print(f"File size: {dst.stat().st_size:,} bytes")


In [ ]:
# Senescence calibration: clipped margin dictionary learning on the Allen atlas.
# Colab: select an A100, upload margin_dictionary_learning.py, and run the
# Drive-copy cell above. This cell uses the prepared atlas without preprocessing.
from pathlib import Path
import gc
import hashlib
import importlib
import json
import math
import sys
import time

import h5py
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import torch
import triton
import triton.language as tl
from IPython.display import display
from tqdm.auto import tqdm

sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'margin_dictionary_learning.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place margin_dictionary_learning.py in the working folder or an ancestor.')
sys.path.insert(0, str(ROOT))
import margin_dictionary_learning
# A re-uploaded shared module must also replace Colab's cached import.
importlib.reload(margin_dictionary_learning)
from margin_dictionary_learning import (
    _FreezingGramFISTA, _audit_sparse_codes, _csr, _dictionary_step, _full_precision,
)

# Editable starting values, not calibrated recommendations. The solver calls
# lambda "alpha". Every candidate in a comparison receives N_UPDATES updates.
RUN_FITS, RESUME_FITS = True, True
ATLAS_PATH = Path('/content/senescence_atlas_processed.h5')
OUTPUT_DIR = Path('/content/drive/MyDrive/senescence_run_20pct/results')
TRAIN_FRACTION, N_VALID = 0.2, 16384
K_CALIBRATION, K_GRID, K_STEP = 200, [120, 160], 40
K_MAX = 800  # Largest K the search may try; bounded by 40-GB GPU memory.
LAMBDA_GRID, LAMBDA_STEP = [0.03, 0.04, 0.05, 0.06], 0.01
Q_GRID, Q_STEP = [0.85, 0.90, 0.95], 0.05
N_UPDATES, LEARNING_RATE = 500, 0.1
CODE_TOL, CODE_MAX_ITER, RECORD_EVERY = 1e-4, 20000, 100
# Initial coding at K around 200 can need more steps; certified solves stop early.
SAMPLE_SEED, GENE_SPLIT_SEED, INITIALIZATION_SEED = 2026, 2027, 1
DEVICE, LOAD_BLOCK_ROWS, AUDIT_ROWS = 'cuda', 4096, 256
CSV_PATH = OUTPUT_DIR / 'calibration.csv'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
EXPERIMENT_PATH = OUTPUT_DIR / 'experiment.npz'
MODEL_SHA256 = hashlib.sha256((ROOT / 'margin_dictionary_learning.py').read_bytes()).hexdigest()
PROTOCOL_VERSION = 'senescence_masked_genes_v1'
progress, records, fit_cache, fit_number = None, [], {}, 0


def load_sample(path, train_fraction, n_valid, seed, block_rows=4096, bar=None):
    """Read selected CSR rows; the complete compressed atlas stays on disk."""
    with h5py.File(path, 'r', rdcc_nbytes=32 * 2**20) as atlas:
        if atlas.attrs.get('format') != 'senescence-csr-v1' or not atlas.attrs.get('complete', False):
            raise ValueError('Expected a completed senescence-csr-v1 atlas.')
        manifest = json.loads(atlas.attrs['manifest_json'])
        expected = dict(cell_fraction=0.001, min_animals=3, target_sum=10000.0,
                        log='natural_log1p', gene_sd_scaling=False, centering=False,
                        positive_variance=True, output_dtype='float32',
                        global_scale='median_cell_l2_after_gene_filter')
        if any(manifest['parameters'].get(k) != v for k, v in expected.items()):
            raise ValueError('Atlas preprocessing differs from the agreed representation.')
        identity = json.dumps(manifest['identity'], sort_keys=True, separators=(',', ':'))
        if (hashlib.sha256(identity.encode()).hexdigest() != manifest['fingerprint']
                or manifest['identity']['parameters'] != manifest['parameters']):
            raise ValueError('Atlas preprocessing fingerprint is inconsistent.')
        n, p = map(int, manifest['shape'])
        n_train = math.ceil(train_fraction * n)
        if not 0 < train_fraction < 1 or min(n_train, n_valid, block_rows) < 1 or n_train + n_valid > n:
            raise ValueError('Use positive disjoint training/validation samples within the atlas.')
        group = atlas['X']
        if tuple(group.attrs['shape']) != (n, p) or p < 2:
            raise ValueError('Invalid atlas dimensions.')
        if any(group[k].dtype != np.dtype(v) for k, v in
               [('data', 'float32'), ('indices', 'int32'), ('indptr', 'int64')]):
            raise ValueError('Unexpected atlas CSR dtypes.')
        indptr = group['indptr'][:]
        if (len(indptr) != n + 1 or indptr[0] != 0 or np.any(np.diff(indptr) < 0)
                or indptr[-1] != len(group['data']) or len(group['indices']) != len(group['data'])):
            raise ValueError('Invalid atlas CSR pointers.')
        # Fix validation membership first. Larger training samples extend a
        # prefix of the same ordering rather than drawing a different sample.
        ordering = np.random.default_rng(seed).permutation(n)
        valid_rows, train_rows = ordering[:n_valid], ordering[n_valid:n_valid + n_train]
        selected = np.r_[train_rows, valid_rows]
        counts = indptr[selected + 1] - indptr[selected]
        sample_ptr = np.r_[np.int64(0), np.cumsum(counts, dtype=np.int64)]
        if sample_ptr[-1] >= 2**31:
            raise ValueError('The calibration sample exceeds the CUDA CSR int32 index limit.')
        values = np.empty(int(sample_ptr[-1]), dtype=np.float32)
        columns = np.empty(int(sample_ptr[-1]), dtype=np.int32)
        cell_ids = np.empty(len(selected), dtype=object)
        order = np.argsort(selected)
        sorted_rows = selected[order]
        if bar is not None:
            bar.reset(total=n)
            bar.set_description('Loading atlas sample')
        for start in range(0, n, block_rows):
            stop = min(n, start + block_rows)
            lo, hi = np.searchsorted(sorted_rows, [start, stop])
            if hi > lo:
                first, last = int(indptr[start]), int(indptr[stop])
                block_values, block_columns = group['data'][first:last], group['indices'][first:last]
                slots = order[lo:hi]
                cell_ids[slots] = atlas['obs/cell_ids'].asstr()[start:stop][selected[slots] - start]
                for slot in slots:
                    row = int(selected[slot])
                    a, b = int(indptr[row]) - first, int(indptr[row + 1]) - first
                    dest = slice(int(sample_ptr[slot]), int(sample_ptr[slot + 1]))
                    values[dest], columns[dest] = block_values[a:b], block_columns[a:b]
            if bar is not None:
                bar.update(stop - start)
        if (not np.isfinite(values).all() or np.any(values < 0)
                or np.any(columns < 0) or np.any(columns >= p)):
            raise ValueError('Invalid sampled expression values or columns.')
        sampled = sparse.csr_matrix((values, columns, sample_ptr.astype(np.int32)),
                                    shape=(len(selected), p))
        sampled.check_format(full_check=True)
        gene_ids = atlas['var/gene_ids'].asstr()[:].astype(str)
        if (len(gene_ids) != p or np.any(atlas['var/detected_cells'][:] < math.ceil(0.001 * n))
                or np.any(atlas['var/detected_animals'][:] < 3)
                or np.any(atlas['var/log_variance'][:] <= 0)):
            raise ValueError('Retained genes violate the declared support filter.')
        return dict(X_train=sampled[:n_train].tocsr(), X_valid=sampled[n_train:].tocsr(),
                    train_rows=train_rows, valid_rows=valid_rows,
                    train_ids=np.asarray(cell_ids[:n_train], dtype=str),
                    valid_ids=np.asarray(cell_ids[n_train:], dtype=str), gene_ids=gene_ids,
                    atlas_fingerprint=manifest['fingerprint'])


def text_hash(values):
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode() + b'\0')
    return digest.hexdigest()


def save_npz(path, **arrays):
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('wb') as handle:
        np.savez(handle, **arrays)
    temporary.replace(path)


def save_records():
    temporary = CSV_PATH.with_suffix('.csv.tmp')
    # Keep a readable header even if interrupted before the first checkpoint.
    frame = pd.DataFrame(records) if records else pd.DataFrame(columns=[
        'experiment_id', 'stage', 'fit_id', 'k', 'lambda', 'q', 'prediction_error',
        'mean_active_atoms', 'iteration', 'elapsed_seconds', 'status', 'selected'])
    frame.to_csv(temporary, index=False)
    temporary.replace(CSV_PATH)


def fit_key(k, lam, q):
    return f'k{k}_lambda{lam:.12g}_q{q:.12g}'


def record_trace(stage, round_index, trace):
    keys = {(stage, row['fit_id'], row['iteration']) for row in trace}
    records[:] = [row for row in records
                  if (row['stage'], row['fit_id'], row['iteration']) not in keys]
    records.extend(dict(row, stage=stage, round=round_index, selected=False) for row in trace)
    save_records()


def weight_rule(margins, q):
    positive = margins[margins > 0]
    if positive.numel() == 0:
        return torch.zeros_like(margins)
    # Exact linear quantile from two order statistics. This also works above
    # torch.quantile's input-size limit; the cap uses the entire sample.
    rank = q * (positive.numel() - 1)
    low, high = math.floor(rank), math.ceil(rank)
    cap = positive.kthvalue(low + 1).values
    if high != low:
        cap = torch.lerp(cap, positive.kthvalue(high + 1).values, rank - low)
    return margins.clamp_min(0).clamp_max(cap)


def row_norm2(x, device):
    """Float64 row squared norms of CSR data, as _TorchGramFISTA computes them."""
    norms = np.zeros(x.shape[0], dtype=np.float64)
    nonempty = np.diff(x.indptr) > 0
    if x.nnz:
        norms[nonempty] = np.add.reduceat(np.square(x.data, dtype=np.float64), x.indptr[:-1][nonempty])
    return torch.as_tensor(norms, device=device)


@torch.no_grad()
def masked_prediction(x, s, alpha, halves, device):
    """Infer from one gene half, predict the other, then swap; zeros count."""
    norm2 = float(np.square(x.data, dtype=np.float64).sum())
    if norm2 <= 0:
        raise ValueError('Validation expression has zero squared norm.')
    total, gap, margin, minimum_gap, steps = 0.0, 0.0, 0.0, math.inf, 0
    with _full_precision():
        for observed, hidden in (halves, halves[::-1]):
            xo, xh = x[:, observed].tocsr(), x[:, hidden].tocsr()
            so, sh = s[:, observed].contiguous(), s[:, hidden].contiguous()
            # Restricted atoms retain their original scale. The penalty tracks
            # the observed fraction of the full squared-loss objective.
            observed_alpha = alpha * len(observed) / x.shape[1]
            initial = s.new_zeros((x.shape[0], len(s)))
            cross, norms = torch.sparse.mm(_csr(xo, device), so.T.contiguous()), row_norm2(xo, device)
            solver = TRAINING_SOLVER(so, initial, observed_alpha, CODE_TOL, CODE_MAX_ITER,
                                     cross=cross, norm2=norms)
            a, _, _, metrics = solver.solve(so, initial, cross=cross, norm2=norms)
            a = a.clone()
            del solver
            _, certificate = _audit_sparse_codes(
                xo, so, a, observed_alpha, CODE_TOL, CODE_MAX_ITER,
                used_steps=metrics['inner_steps'], block_rows=AUDIT_ROWS)
            gap = max(gap, certificate['relative_gap'])
            margin = max(margin, certificate['relative_margin_error'])
            minimum_gap = min(minimum_gap, certificate['minimum_relative_gap'])
            steps = max(steps, certificate['inner_steps'])
            for first in range(0, x.shape[0], AUDIT_ROWS):
                last = min(x.shape[0], first + AUDIT_ROWS)
                target = torch.as_tensor(xh[first:last].toarray(), dtype=s.dtype, device=device)
                residual = target - a[first:last] @ sh
                total += residual.double().square().sum().item()
            del a, xo, xh, so, sh, cross, norms
    return dict(prediction_error=total / norm2, validation_relative_gap=gap,
                validation_relative_margin_error=margin,
                validation_minimum_relative_gap=minimum_gap, validation_inner_steps=steps)


def synchronize():
    if torch.device(DEVICE).type == 'cuda':
        torch.cuda.synchronize()


def completed_result(trace):
    row = trace[-1]
    if row['iteration'] != N_UPDATES or row['status'] != 'complete' or not row['training_audited']:
        raise ValueError('A candidate must finish the common budget and its numerical audit.')
    for name in ('prediction_error', 'training_objective'):
        if not math.isfinite(row[name]) or row[name] < 0:
            raise ValueError(f'Invalid candidate score: {name}.')
    for prefix in ('training', 'validation'):
        values = [row[prefix + suffix] for suffix in
                  ('_relative_gap', '_relative_margin_error', '_minimum_relative_gap')]
        if (not all(math.isfinite(v) for v in values)
                or max(values[:2]) > CODE_TOL or values[2] < -CODE_TOL):
            raise ValueError(f'Invalid {prefix} certificate in the saved candidate.')
    return row.copy()


def progress_text(trace, inner_steps):
    """Latest training solve; E and cosine belong to the displayed checkpoint."""
    row = trace[-1]
    change = ''
    if len(trace) > 1 and trace[-2]['prediction_error'] > 0:
        change = f" ({100 * (row['prediction_error'] / trace[-2]['prediction_error'] - 1):+.2f}%)"
    cosine = row['mean_atom_cosine_previous']
    cosine_text = '--' if cosine is None else f'{cosine:.4f}'
    return (f"inner={inner_steps} | E@{row['iteration']}={row['prediction_error']:.4f}"
            f"{change} | cos={cosine_text}")


@triton.jit
def _fista_step_kernel(yg_ptr, cross_ptr, y_ptr, a_ptr, lip_ptr, beta_ptr, alpha, n,
                       BLOCK: tl.constexpr):
    i = tl.program_id(0).to(tl.int64) * BLOCK + tl.arange(0, BLOCK)
    mask = i < n
    yg, cross = tl.load(yg_ptr + i, mask=mask), tl.load(cross_ptr + i, mask=mask)
    y, a = tl.load(y_ptr + i, mask=mask), tl.load(a_ptr + i, mask=mask)
    t = y + tl.div_rn(cross - yg - alpha, tl.load(lip_ptr))
    a_next = tl.where(t <= 0, 0.0, t)
    tl.store(a_ptr + i, a_next, mask=mask)
    tl.store(y_ptr + i, a_next + tl.load(beta_ptr) * (a_next - a), mask=mask)


class FusedFreezingGramFISTA(_FreezingGramFISTA):
    """_FreezingGramFISTA with the elementwise part of each step in one kernel."""

    def _step(self, a, y, cross, beta):
        _fista_step_kernel[(triton.cdiv(a.numel(), 1024),)](
            y @ self.gram, cross, y, a, self.lip, beta, self.alpha, a.numel(),
            BLOCK=1024, enable_fp_fusion=False)
        return a, y


@torch.no_grad()
def fused_step_matches():
    """Check that the fused step reproduces the PyTorch step bit for bit."""
    generator = torch.Generator(device=DEVICE).manual_seed(0)
    s = torch.randn(200, 512, device=DEVICE, generator=generator)
    s /= s.norm(dim=1, keepdim=True)
    a = torch.rand(3001, 200, device=DEVICE, generator=generator)
    cross = torch.randn(3001, 200, device=DEVICE, generator=generator)
    norm2 = cross.double().square().sum(1)
    results = []
    with _full_precision():
        for solver_class in (_FreezingGramFISTA, FusedFreezingGramFISTA):
            solver = solver_class(s, a, 0.05, CODE_TOL, 20, cross=cross, norm2=norm2)
            codes, y = a.clone(), a.clone()
            for j in range(20):
                codes, y = solver._step(codes, y, cross, solver.schedule[j])
            results.append((codes, y))
    return all(torch.equal(x, z) for x, z in zip(*results))


def prepare_training_products():
    """Keep training cells on the GPU: dense float32 if it fits, otherwise CSR."""
    global TRAIN_NORM2, TRAIN_PRODUCTS, TRAIN_TIMES, TRAIN_T_TIMES, TRAINING_SOLVER
    n, p = X_TRAIN.shape
    TRAIN_NORM2 = row_norm2(X_TRAIN, DEVICE)
    needed = 4 * n * p + max(6 * 2**30, 100 * n * (max(K_CALIBRATION, *K_GRID) + 2 * K_STEP))
    free = torch.cuda.mem_get_info()[0] if torch.device(DEVICE).type == 'cuda' else math.inf
    if needed <= free:
        dense = torch.empty((n, p), dtype=torch.float32, device=DEVICE)
        for start in range(0, n, LOAD_BLOCK_ROWS):
            dense[start:start + LOAD_BLOCK_ROWS] = torch.as_tensor(
                X_TRAIN[start:start + LOAD_BLOCK_ROWS].toarray())
        TRAIN_PRODUCTS = 'dense'
        TRAIN_TIMES, TRAIN_T_TIMES = (lambda s: dense @ s.T), (lambda w: dense.T @ w)
    else:
        x, xt = _csr(X_TRAIN, DEVICE), _csr(X_TRAIN.T.tocsr(), DEVICE)
        TRAIN_PRODUCTS = 'sparse'
        TRAIN_TIMES = lambda s: torch.sparse.mm(x, s.T.contiguous())
        TRAIN_T_TIMES = lambda w: torch.sparse.mm(xt, w)
    fused = torch.device(DEVICE).type == 'cuda' and fused_step_matches()
    TRAINING_SOLVER = FusedFreezingGramFISTA if fused else _FreezingGramFISTA
    print(f'Training products: {TRAIN_PRODUCTS} ({needed / 2**30:.1f} GiB needed, '
          f'{free / 2**30:.1f} GiB free); solver: {TRAINING_SOLVER.__name__}.')


def run_fit(stage, round_index, k, lam, q):
    """Continue S, A and Adam to the common budget; checkpoint before reporting."""
    global fit_number
    key = fit_key(k, lam, q)
    path = CHECKPOINT_DIR / (EXPERIMENT_ID[:16] + '_' + key + '.npz')
    if key in fit_cache:
        result = completed_result(fit_cache[key])
        record_trace(stage, round_index, fit_cache[key])
        return result
    state, arrays = None, None
    if RESUME_FITS and path.exists():
        with np.load(path, allow_pickle=False) as saved:
            state = json.loads(str(saved['metadata_json']))
            if state['experiment_id'] != EXPERIMENT_ID or state['fit_id'] != key:
                raise ValueError('Incompatible fit checkpoint; use a fresh output directory.')
            if state['iteration'] > N_UPDATES:
                raise ValueError('Cannot rewind the latest checkpoint; increase N_UPDATES or start fresh.')
            if state['iteration'] < N_UPDATES:
                arrays = {name: saved[name].copy() for name in ('S', 'A', 'first', 'second')}
        if state['iteration'] == N_UPDATES:
            result = completed_result(state['trace'])
            fit_cache[key] = state['trace']
            record_trace(stage, round_index, state['trace'])
            return result
    if k > len(INITIAL_ROWS):
        raise ValueError('K exceeds the number of nonzero training cells.')
    fit_number += 1
    if progress is not None:
        progress.set_postfix_str('initial coding', refresh=False)
        progress.reset(total=N_UPDATES)
        progress.set_description(f'{stage} fit {fit_number}: K={k}, lambda={lam:g}, q={q:g}')
    session_start = time.perf_counter()
    if torch.device(DEVICE).type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    elapsed_before = 0.0 if state is None else state['trace'][-1]['elapsed_seconds']
    trace = [] if state is None else list(state['trace'])
    step = 0 if state is None else state['iteration']
    training_seconds = 0.0 if state is None else state['training_seconds']
    validation_seconds = 0.0 if state is None else state['validation_seconds']
    audit_seconds = 0.0 if state is None else state['audit_seconds']
    if arrays is None:
        initial = torch.as_tensor(X_TRAIN[INITIAL_ROWS[:k]].toarray(), dtype=torch.float64, device=DEVICE)
        s = (initial / initial.norm(dim=1, keepdim=True)).float()
        s /= s.norm(dim=1, keepdim=True)
        a = s.new_zeros((X_TRAIN.shape[0], k))
        first, second = torch.zeros_like(s), torch.zeros_like(s[:, :1])
    else:
        s, a, first, second = (torch.as_tensor(arrays[name], device=DEVICE).clone()
                               for name in ('S', 'A', 'first', 'second'))
        del arrays
    reference_s, reference_a = s.cpu().numpy().copy(), a.cpu().numpy().copy()
    window_steps, window_seconds, window_inner, window_rows = 0, 0.0, 0, 0.0
    solver = None
    try:
        with torch.no_grad(), _full_precision():
            synchronize()
            tic = time.perf_counter()
            cross = TRAIN_TIMES(s)
            # Saved codes were accepted at half the tolerance on subsets of rows; the
            # full-matrix recheck rounds slightly differently, so recheck them at CODE_TOL.
            solver = TRAINING_SOLVER(s, a, lam, CODE_TOL, CODE_MAX_ITER, cross=cross, norm2=TRAIN_NORM2,
                                     freeze_margin=0.5 if state is None else 1.0)
            a, correlation, residual2, metrics = solver.solve(s, a, cross=cross, norm2=TRAIN_NORM2)
            if state is not None and metrics['inner_steps']:
                raise RuntimeError('Saved codes require refinement; the saved trajectory cannot be resumed unchanged.')
            if state is not None:
                solver = TRAINING_SOLVER(s, a, lam, CODE_TOL, CODE_MAX_ITER, cross=cross, norm2=TRAIN_NORM2)
            del cross
            synchronize()
            training_seconds += time.perf_counter() - tic
            if progress is not None:
                progress.update(step)

            def checkpoint():
                nonlocal reference_s, reference_a, validation_seconds, audit_seconds
                nonlocal window_steps, window_seconds, window_inner, window_rows
                snapshot_s, snapshot_a = s.cpu().numpy().copy(), a.cpu().numpy().copy()
                sd, rd = snapshot_s.astype(np.float64), reference_s.astype(np.float64)
                cosine = np.einsum('ij,ij->i', sd, rd) / (np.linalg.norm(sd, axis=1) * np.linalg.norm(rd, axis=1))
                train_sse, certified = residual2, metrics.copy()
                reported_a = a
                if step == N_UPDATES:
                    tic = time.perf_counter()
                    reported_a = a.clone()  # Never replace the optimizer's pre-audit A.
                    train_sse, certified = _audit_sparse_codes(
                        X_TRAIN, s, reported_a, lam, CODE_TOL, CODE_MAX_ITER,
                        used_steps=metrics['inner_steps'], block_rows=AUDIT_ROWS)
                    synchronize()
                    audit_seconds += time.perf_counter() - tic
                tic = time.perf_counter()
                prediction = masked_prediction(X_VALID, s, lam, GENE_HALVES, DEVICE)
                synchronize()
                validation_seconds += time.perf_counter() - tic
                row = dict(experiment_id=EXPERIMENT_ID, fit_id=key, k=k, q=q,
                           learning_rate=LEARNING_RATE, seed=INITIALIZATION_SEED,
                           n_train=X_TRAIN.shape[0], n_valid=X_VALID.shape[0], iteration=step,
                           training_objective=(0.5 * train_sse + lam * reported_a.double().sum().item()) / X_TRAIN.shape[0],
                           mean_active_atoms=float((reported_a > 0).sum().item() / X_TRAIN.shape[0]),
                           used_atoms=int((reported_a > 0).any(dim=0).sum().item()),
                           mean_atom_cosine_previous=float(np.clip(cosine, -1, 1).mean()) if window_steps else None,
                           min_atom_cosine_previous=float(np.clip(cosine, -1, 1).min()) if window_steps else None,
                           relative_activity_change=float(np.linalg.norm(snapshot_a.astype(np.float64) - reference_a) /
                                                          max(np.linalg.norm(reference_a.astype(np.float64)), 1e-12)) if window_steps else None,
                           mean_inner_steps=window_inner / window_steps if window_steps else 0.0,
                           mean_row_steps=window_rows / window_steps if window_steps else 0.0,
                           seconds_per_update=window_seconds / window_steps if window_steps else 0.0,
                           training_relative_gap=certified['relative_gap'],
                           training_relative_margin_error=certified['relative_margin_error'],
                           training_minimum_relative_gap=certified['minimum_relative_gap'],
                           training_audited=step == N_UPDATES, training_seconds=training_seconds,
                           validation_seconds=validation_seconds, audit_seconds=audit_seconds,
                           elapsed_seconds=elapsed_before + time.perf_counter() - session_start,
                           peak_gpu_gib=torch.cuda.max_memory_allocated() / 2**30 if s.is_cuda else 0.0,
                           status='complete' if step == N_UPDATES else 'checkpoint', **prediction)
                row['lambda'] = lam
                trace.append(row)
                metadata = dict(experiment_id=EXPERIMENT_ID, fit_id=key, iteration=step, trace=trace,
                                training_seconds=training_seconds, validation_seconds=validation_seconds,
                                audit_seconds=audit_seconds)
                save_npz(path, S=snapshot_s, A=snapshot_a,
                         first=first.cpu().numpy(), second=second.cpu().numpy(),
                         metadata_json=np.array(json.dumps(metadata, allow_nan=False)))
                record_trace(stage, round_index, trace)
                reference_s, reference_a = snapshot_s, snapshot_a
                window_steps, window_seconds, window_inner, window_rows = 0, 0.0, 0, 0.0

            if state is None:
                checkpoint()
            if progress is not None:
                progress.set_postfix_str(progress_text(trace, metrics['inner_steps']))
            while step < N_UPDATES:
                synchronize()
                tic = time.perf_counter()
                weights = weight_rule(a + correlation - lam, q)
                gradient = ((weights.T @ a) @ s - TRAIN_T_TIMES(weights).T) / (
                    X_TRAIN.shape[0] * X_TRAIN.shape[1])
                s = _dictionary_step(s, weights, None, first, second, step + 1,
                                     LEARNING_RATE, gradient=gradient)
                a, correlation, residual2, metrics = solver.solve(
                    s, a, cross=TRAIN_TIMES(s), norm2=TRAIN_NORM2)
                synchronize()
                duration = time.perf_counter() - tic
                step += 1
                training_seconds += duration
                window_steps += 1
                window_seconds += duration
                window_inner += metrics['inner_steps']
                window_rows += metrics['row_steps']
                # Count the completed update before validation; E stays labelled
                # with its previous checkpoint until a new score is available.
                if progress is not None:
                    progress.set_postfix_str(progress_text(trace, metrics['inner_steps']), refresh=False)
                    progress.update(1)
                if step % RECORD_EVERY == 0 or step == N_UPDATES:
                    checkpoint()
                    if progress is not None:
                        progress.set_postfix_str(progress_text(trace, metrics['inner_steps']))
            fit_cache[key] = trace
            return completed_result(trace)
    finally:
        del solver
        gc.collect()


def select_lambda_q(rows):
    if not rows or any(not math.isfinite(row['prediction_error']) for row in rows):
        raise ValueError('Selection requires finite prediction errors for every candidate.')
    return min(rows, key=lambda row: (row['prediction_error'], row['lambda'], -row['q']))


def boundary_extensions(lambdas, quantiles, chosen, improved):
    new_lambdas, new_q = set(), set()
    if improved:
        lower = round(min(lambdas) - LAMBDA_STEP, 12)
        if chosen['lambda'] == min(lambdas) and lower > 0:
            new_lambdas.add(lower)
        if chosen['lambda'] == max(lambdas):
            new_lambdas.add(round(max(lambdas) + LAMBDA_STEP, 12))
        if chosen['q'] == min(quantiles) and chosen['q'] > 0:
            new_q.add(round(max(0.0, chosen['q'] - Q_STEP), 12))
        if chosen['q'] == max(quantiles) and chosen['q'] < 1:
            new_q.add(round(min(1.0, chosen['q'] + Q_STEP), 12))
    return sorted(new_lambdas - set(lambdas)), sorted(new_q - set(quantiles))


def select_k(rows):
    if not rows or any(not math.isfinite(row['prediction_error']) for row in rows):
        raise ValueError('Selection requires finite prediction errors for every candidate.')
    best = min(row['prediction_error'] for row in rows)
    return min(row['k'] for row in rows if row['prediction_error'] <= 1.01 * best)


def mark_selected(stage, chosen):
    for row in records:
        if row['stage'] == stage:
            row['selected'] = row['fit_id'] == chosen['fit_id'] and row['iteration'] == N_UPDATES
    save_records()


def search_lambda_q():
    lambdas, quantiles = sorted(set(LAMBDA_GRID)), sorted(set(Q_GRID))
    rows, previous_best, round_index = {}, None, 0
    while True:
        for lam in lambdas:
            for q in quantiles:
                if (lam, q) not in rows:
                    rows[lam, q] = run_fit('lambda_q', round_index, K_CALIBRATION, lam, q)
        chosen = select_lambda_q(list(rows.values()))
        best = chosen['prediction_error']
        # As in the other notebooks: try an edge once, then require improvement.
        extra_lambdas, extra_q = boundary_extensions(
            lambdas, quantiles, chosen, previous_best is None or best < previous_best)
        if not extra_lambdas and not extra_q:
            mark_selected('lambda_q', chosen)
            return chosen
        lambdas, quantiles = sorted(set(lambdas + extra_lambdas)), sorted(set(quantiles + extra_q))
        previous_best, round_index = best, round_index + 1


def search_k(lam, q):
    candidates = sorted(set(K_GRID))
    rows, previous_best, round_index = {}, None, 0
    while True:
        for k in candidates:
            if k not in rows:
                rows[k] = run_fit('K', round_index, k, lam, q)
        best_row = min(rows.values(), key=lambda row: (row['prediction_error'], row['k']))
        best = best_row['prediction_error']
        pending = []
        if previous_best is None or best < previous_best:
            lower, upper = min(candidates) - K_STEP, max(candidates) + K_STEP
            if best_row['k'] == min(candidates) and lower > 0:
                pending.append(lower)
            if best_row['k'] == max(candidates) and upper <= K_MAX:
                pending.append(upper)
        if not pending:
            chosen = rows[select_k(list(rows.values()))]
            mark_selected('K', chosen)
            return chosen
        candidates = sorted(set(candidates + pending))
        previous_best, round_index = best, round_index + 1


def initialize_experiment(prepared, load_seconds=None):
    """One shared record identifies data, splits and the numerical procedure."""
    global EXPERIMENT_ID, GENE_HALVES, INITIAL_ROWS
    p = len(prepared['gene_ids'])
    genes = np.random.default_rng(GENE_SPLIT_SEED).permutation(p)
    GENE_HALVES = tuple(np.sort(part) for part in (genes[:p // 2], genes[p // 2:]))
    nonzero = np.flatnonzero(np.diff(prepared['X_train'].indptr) > 0)
    INITIAL_ROWS = np.random.default_rng(INITIALIZATION_SEED).permutation(nonzero)
    identity = dict(protocol=PROTOCOL_VERSION, atlas_fingerprint=prepared['atlas_fingerprint'],
                    train_ids=text_hash(prepared['train_ids']), valid_ids=text_hash(prepared['valid_ids']),
                    gene_ids=text_hash(prepared['gene_ids']), gene_half=text_hash(GENE_HALVES[0]),
                    sample_seed=SAMPLE_SEED, gene_split_seed=GENE_SPLIT_SEED,
                    initialization_seed=INITIALIZATION_SEED, learning_rate=LEARNING_RATE,
                    code_tol=CODE_TOL, code_max_iter=CODE_MAX_ITER, audit_rows=AUDIT_ROWS,
                    model_sha256=MODEL_SHA256, training_products=TRAIN_PRODUCTS,
                    device=str(DEVICE), torch_version=torch.__version__)
    EXPERIMENT_ID = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    if RESUME_FITS and EXPERIMENT_PATH.exists():
        with np.load(EXPERIMENT_PATH, allow_pickle=False) as saved:
            old = json.loads(str(saved['metadata_json']))
        if old['experiment_id'] != EXPERIMENT_ID:
            raise ValueError('Saved experiment differs; use a fresh OUTPUT_DIR or RESUME_FITS=False.')
    metadata = dict(experiment_id=EXPERIMENT_ID, identity=identity, loading_seconds=load_seconds,
                    software=dict(python=sys.version, numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__,
                                  h5py=h5py.__version__, torch=torch.__version__, cuda=torch.version.cuda),
                    settings=dict(train_fraction=TRAIN_FRACTION, n_valid=N_VALID, k=K_CALIBRATION,
                                  lambda_grid=LAMBDA_GRID, q_grid=Q_GRID, k_grid=K_GRID,
                                  lambda_step=LAMBDA_STEP, q_step=Q_STEP, k_step=K_STEP, k_max=K_MAX,
                                  updates=N_UPDATES, record_every=RECORD_EVERY))
    save_npz(EXPERIMENT_PATH, metadata_json=np.array(json.dumps(metadata, allow_nan=False)),
             train_rows=prepared['train_rows'], valid_rows=prepared['valid_rows'],
             train_ids=prepared['train_ids'], valid_ids=prepared['valid_ids'],
             gene_ids=prepared['gene_ids'], observed_genes=GENE_HALVES[0], hidden_genes=GENE_HALVES[1])


# Validation cells share the atlas's frozen preprocessing and may share mice
# with training cells: this is internal prediction, not new-animal validation.
# Cosines below describe atom movement between checkpoints, not recovery of
# known biological truth. N_UPDATES is a budget, not a convergence claim.
if RUN_FITS:
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Google Drive using the first cell before running calibration.')
    if torch.device(DEVICE).type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Select a Colab GPU runtime before running calibration.')
    if not ATLAS_PATH.is_file():
        raise FileNotFoundError(f'Copy the prepared compressed atlas to {ATLAS_PATH}.')
    if (not LAMBDA_GRID or min(LAMBDA_GRID) <= 0 or not np.isfinite(LAMBDA_GRID).all()
            or not Q_GRID or not all(0 <= q <= 1 for q in Q_GRID)
            or not K_GRID or any(int(k) != k or k < 1 for k in [K_CALIBRATION, *K_GRID])
            or any(not np.isfinite(v) or v <= 0 for v in [LAMBDA_STEP, Q_STEP, LEARNING_RATE, CODE_TOL])
            or any(int(v) != v or v < 1 for v in [K_STEP, N_UPDATES, CODE_MAX_ITER, RECORD_EVERY, N_VALID])):
        raise ValueError('Check the grids, positive step sizes, budgets and numerical settings.')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    torch.set_num_threads(2)
    with tqdm(total=1, desc='Loading atlas', mininterval=1) as progress:
        load_start = time.perf_counter()
        prepared = load_sample(ATLAS_PATH, TRAIN_FRACTION, N_VALID, SAMPLE_SEED, LOAD_BLOCK_ROWS, progress)
        X_TRAIN, X_VALID = prepared['X_train'], prepared['X_valid']
        prepare_training_products()
        initialize_experiment(prepared, time.perf_counter() - load_start)
        records = pd.read_csv(CSV_PATH).to_dict('records') if RESUME_FITS and CSV_PATH.exists() else []
        if any(row['experiment_id'] != EXPERIMENT_ID for row in records):
            raise ValueError('The CSV belongs to a different experiment.')
        # Recompute both selections at this budget; old choices cannot stand in
        # for an incomplete new grid. Candidate files remain reusable.
        records[:] = []
        save_records()
        selected_pair = search_lambda_q()
        selected_model = search_k(selected_pair['lambda'], selected_pair['q'])
    progress = None

if not CSV_PATH.is_file():
    raise FileNotFoundError('No calibration.csv exists; set RUN_FITS=True to perform calibration.')
table = pd.read_csv(CSV_PATH)
table = table.loc[table.iteration.eq(N_UPDATES) & table.status.eq('complete')]
table = table.sort_values(['stage', 'k', 'lambda', 'q'])
columns = ['stage', 'k', 'lambda', 'q', 'prediction_error', 'mean_active_atoms',
           'iteration', 'elapsed_seconds', 'selected']
display(table[columns].rename(columns={'k': 'K', 'iteration': 'updates', 'elapsed_seconds': 'seconds'})
        .style.format({'lambda': '{:.3g}', 'q': '{:.3g}', 'prediction_error': '{:.6f}',
                       'mean_active_atoms': '{:.2f}', 'seconds': '{:.1f}'}))